# Cutoff-aware policy vs. swap-asap, under delayed classical information

For a grid of `(p, t_cut)` values -- swept once per swap success probability `p_s` -- this
notebook measures one hand-written policy against the baseline on the **delayed-information**
repeater chain (`info_delay_setup.jl`), where each node
sees only its own slots and learns everything else through messages that travel one hop per
timestep.

- **swap-asap** — the baseline: swap the moment both slots are *believed* to hold a link. It
  cannot tell a live link from a hanging qubit, so it burns good links against dead ones and
  pushes the damage further down the chain.
- **cutoff-aware** (`CutoffAwareSwap(0)`) — swap-asap plus one inference: **discard a slot once
  the flooded ledger projects its far end past its own retention time.** The far end must
  already have been swept, so this slot is hanging; freeing it now lets link generation restart
  on that edge instead of feeding a doomed swap that would hang somebody else's qubit too.

That is the whole policy. It is the smallest thing a node can do with delayed information that
swap-asap cannot do at all, which is what makes it worth measuring on its own.

### How the inference works, and how it can be wrong

Each node keeps a flooded ledger: every node re-asserts the age of its own slots every timestep
and passes the whole table to both neighbours, so node `i`'s entry for a qubit at node `j` is
`j`'s truth as of `t - |i-j|`. The *projected* age of a remote qubit is the last age heard plus
the time since. When that projection passes `t_cut`, the far end must have been discarded.

The projection is wrong exactly when something else happened to that qubit in the meantime --
most importantly when it was **swapped** rather than cut, in which case the node discards a link
that was alive. That is the risk the policy is taking, and the grid below is where it pays off
and where it does not.

### On `MARGIN`

`CutoffAwareSwap` has a second, optional rule: don't swap links within `margin` of being cut
anyway. **It defaults to 0 -- off -- because that rule is actively harmful.** A link at age
exactly `t_cut` is on its final timestep, so swapping it now is the only value it has left;
any `margin > 0` refuses that last-chance swap. At `margin=1` this policy runs up to **40%
worse** than swap-asap at low `p` (21.5 vs 12.9 at `t_cut=2, p=0.4`, measured on an earlier
sweep that reached down to `t_cut=2`). Set `MARGIN` below to see it. Worth knowing before adding any similar "conserve it for later" heuristic.

### Reading the numbers

Unlike `local_policy_heatmap*.ipynb` and `pomdp_policy_heatmap.ipynb`, which evaluate policies
**exactly** by solving the MDP, nothing here can: with delayed information a node's local
observation is not Markov, so there is no small exact model to solve. **Every number is Monte
Carlo**, and the delivery-time distribution is heavy tailed -- a hung qubit blocks its slot for a
full retention time, so episodes pile up at multiples of `t_cut` and the sample standard
deviation runs several times the mean. Cells that do not clear 2 standard errors are drawn
parenthesised and muted.

Delivery-time panels plot `(T_swap-asap − T_policy) / T_policy`, matching the other notebooks,
one panel per `p_s`. Fidelities are still recorded but not plotted in this pass.

In [ ]:
import sys
from pathlib import Path

# Locate src/ (where info_delay_setup.jl / info_delay_solver.py live) regardless
# of whether Jupyter's cwd is src/ itself or the repo root.
_candidates = [Path.cwd(), Path.cwd() / "src"]
SRC_DIR = next((c for c in _candidates if (c / "info_delay_setup.jl").exists()), None)
if SRC_DIR is None:
    raise RuntimeError(
        "Couldn't find info_delay_setup.jl -- run this notebook from the repo's "
        "src/ directory, or from the repo root."
    )
sys.path.insert(0, str(SRC_DIR))

import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Only the Julia bridge is needed here. info_delay_solver.py's JESP solver is not
# used by this notebook -- both policies measured below are built into
# info_delay_setup.jl, so nothing has to be solved first.
from info_delay_solver import run_grid_in_simulator

## Parameters

Nothing is solved here, so the whole cost is simulation. That buys a lot of trials, which
matters: the effect being measured is a few percent against a heavy-tailed distribution, and at
6000 trials most of the grid does not clear 2 standard errors.

The sweep is a `(p, t_cut)` grid repeated once per swap success probability `p_s`, which is what
sets how often a swap fails and leaves a qubit hanging -- the failure mode the cutoff-aware rule
exists to clean up. Six values of `p_s` multiply the simulation cost accordingly.

In [ ]:
N = 5                    # number of nodes in the chain

TRIALS = 20000           # simulated episodes per policy per grid point. Generous on purpose:
                         # the delivery-time sd runs several times the mean, so resolving a
                         # 5% effect needs the sample size. Halve it for a quick look.
TAU = 50.0               # memory depolarization time constant. Only moves fidelity -- the
                         # classical layer never reads it -- so it touches nothing plotted
                         # below, and is kept only so the recorded fidelities stay legible.
F_NEW = 1.0              # fidelity of a freshly generated elementary link
MARGIN = 0               # CutoffAwareSwap's headroom rule. 0 = off (recommended). Set to 1
                         # to reproduce the up-to-40% regression described above.
SIM_SEED = 20240820

P_VALUES = np.round(np.arange(0.3, 0.61, 0.1), 2)     # x-axis: link generation probability
T_CUT_VALUES = list(range(4, 9))                      # y-axis: qubit retention time
P_S_VALUES = np.round(np.arange(0.5, 1.001, 0.1), 2)  # one heatmap per swap success
                         # probability. p_s is the knob that matters most here: a failed swap
                         # is what strands a qubit at the far end, so it manufactures exactly
                         # the situation the cutoff-aware rule is built to detect. p_s = 1.0
                         # (the value the other heatmap notebooks use) is the control -- swaps
                         # never fail there, so most of the delayed-information problem is gone.

print(f"n={N}, tau={TAU}, F_new={F_NEW}, margin={MARGIN}")
print(f"p     in {[float(p) for p in P_VALUES]}")
print(f"t_cut in {T_CUT_VALUES}")
print(f"p_s   in {[float(p_s) for p_s in P_S_VALUES]}")
n_points = len(P_S_VALUES) * len(T_CUT_VALUES) * len(P_VALUES)
print(f"grid: {n_points} points x 2 policies x {TRIALS} trials "
      f"= {2 * n_points * TRIALS:,} episodes")

## Measure

A single Julia process handles the whole grid -- all three axes of it: QuantumSavory takes ~20s
to load, so paying that per grid point would cost more than every simulation in the sweep put
together. Both policies at a point run under the same seed. `paths={}` means no solved policy
tables -- only the two built-in baselines.

In [ ]:
points = [{"n": N, "p": float(p), "p_s": float(p_s), "t_cut": t_cut, "paths": {}}
          for p_s in P_S_VALUES for t_cut in T_CUT_VALUES for p in P_VALUES]

started = time.time()
results = run_grid_in_simulator(points, trials=TRIALS, tau=TAU, F_new=F_NEW,
                                 seed=SIM_SEED, margin=MARGIN)
print(f"measured {len(results)} points in {time.time() - started:.0f}s")

# Axes are (p_s, t_cut, p) throughout: one heatmap per leading index.
shape = (len(P_S_VALUES), len(T_CUT_VALUES), len(P_VALUES))
POLICIES = ("swap-asap", "cutoff-aware")
T = {name: np.empty(shape) for name in POLICIES}
T_sd = {name: np.empty(shape) for name in POLICIES}
F = {name: np.empty(shape) for name in POLICIES}   # recorded, not plotted (no fidelity panel)
timeouts = np.zeros(shape, dtype=int)

by_key = {(round(r["p_s"], 2), r["t_cut"], round(r["p"], 2)): r for r in results}
for k, p_s in enumerate(P_S_VALUES):
    for i, t_cut in enumerate(T_CUT_VALUES):
        for j, p in enumerate(P_VALUES):
            row = by_key[(round(float(p_s), 2), t_cut, round(float(p), 2))]
            for name in POLICIES:
                T[name][k, i, j] = row[name]["delivery_time"]
                T_sd[name][k, i, j] = row[name]["delivery_sd"]
                F[name][k, i, j] = row[name]["fidelity"]
                timeouts[k, i, j] += row[name]["timeouts"]

if timeouts.any():
    print(f"WARNING: {timeouts.sum()} episodes hit max_steps without delivering. "
          f"Those points' averages are conditioned on success and are optimistic.")

print(f"\n{'p_s':>5} {'t_cut':>5} {'p':>5} | {'T swap-asap':>12} {'T cutoff-aw':>12}"
      f" {'delta':>8}")
for k, p_s in enumerate(P_S_VALUES):
    for i, t_cut in enumerate(T_CUT_VALUES):
        for j, p in enumerate(P_VALUES):
            d = T["swap-asap"][k, i, j] - T["cutoff-aware"][k, i, j]
            print(f"{p_s:>5.1f} {t_cut:>5} {p:>5.1f} | {T['swap-asap'][k, i, j]:>12.3f} "
                  f"{T['cutoff-aware'][k, i, j]:>12.3f} {d:>+8.3f}")

In [ ]:
# Fractional improvement over swap-asap: positive means cutoff-aware delivers
# faster, negative means swap-asap actually wins.
ratio = (T["swap-asap"] - T["cutoff-aware"]) / T["cutoff-aware"]

# Two-sample standard error on the delivery-time difference. The delivery-time
# distribution is heavy-tailed, so this is the only thing separating a real effect
# from a lucky batch of episodes.
stderr = np.sqrt(T_sd["swap-asap"] ** 2 / TRIALS + T_sd["cutoff-aware"] ** 2 / TRIALS)
delta = T["swap-asap"] - T["cutoff-aware"]
significant = np.abs(delta) > 2 * stderr

print(f"cells clearing 2 s.e.: {int(significant.sum())}/{significant.size}")
print(f"positive in {int((ratio > 0).sum())}/{ratio.size};  "
      f"best {ratio.max():+.1%}, worst {ratio.min():+.1%}")

# The two patterns worth watching. Across p_s: failed swaps are what strand a qubit,
# so the advantage should thin out as p_s approaches 1 and vanish at the control.
print(f"\n{'p_s':>6} {'mean improvement':>18} {'cells > 0':>11} {'cells > 2 s.e.':>15}")
per_panel = ratio.shape[1] * ratio.shape[2]
for k, p_s in enumerate(P_S_VALUES):
    print(f"{p_s:>6.1f} {ratio[k].mean():>17.1%} "
          f"{int((ratio[k] > 0).sum()):>7}/{per_panel} "
          f"{int(significant[k].sum()):>11}/{per_panel}")

# Across t_cut (pooled over p_s): a longer retention time means more time spent
# holding a qubit whose far end may already be gone.
print(f"\n{'t_cut':>6} {'mean improvement':>18} {'cells > 0':>11} {'cells > 2 s.e.':>15}")
per_row = ratio.shape[0] * ratio.shape[2]
for i, t_cut in enumerate(T_CUT_VALUES):
    print(f"{t_cut:>6} {ratio[:, i].mean():>17.1%} "
          f"{int((ratio[:, i] > 0).sum()):>7}/{per_row} "
          f"{int(significant[:, i].sum()):>11}/{per_row}")

## Visualize

One heatmap per `p_s`, low to high, printed above its panel. Blue beats swap-asap, red loses to
it, gray ties. Cells whose delivery-time difference does not clear 2 standard errors are drawn
**parenthesised and muted** -- at those points the two policies are not measurably different,
however the colour reads.

Every panel shares one colour scale, set by the largest effect anywhere in the sweep, so colours
mean the same thing from panel to panel and the `p_s` trend is readable across them.

Panels plot `(T_swap-asap − T_policy) / T_policy`, matching the other notebooks. There is no
fidelity panel in this pass.

In [ ]:
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"

# Diverging blue <-> red pair with a neutral gray midpoint at zero.
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "blue_gray_red", ["#e34948", "#f0efec", "#2a78d6"]
)


def _text_color_for(rgba):
    """Ink or white, whichever contrasts with a cell's fill."""
    luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
    return INK_PRIMARY if luminance > 0.6 else "#ffffff"


def draw_heatmap(ax, matrix, norm, fmt="{:.2f}", mask=None):
    """`mask` marks the cells that ARE statistically resolved; anything else is
    drawn parenthesised and dimmed."""
    im = ax.imshow(matrix, cmap=DIVERGING_CMAP, norm=norm, aspect="auto", origin="lower")

    ax.set_xticks(range(len(P_VALUES)))
    ax.set_xticklabels([f"{p:.1f}" for p in P_VALUES], color=INK_MUTED)
    ax.set_yticks(range(len(T_CUT_VALUES)))
    ax.set_yticklabels(T_CUT_VALUES, color=INK_MUTED)
    ax.set_xlabel("p  (link generation success probability)", color=INK_SECONDARY)
    ax.set_ylabel("t_cut  (qubit retention time)", color=INK_SECONDARY)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(length=0)

    # A thin surface-color gap between cells, so neighbors read as distinct
    # without drawing a border around every cell.
    ax.set_xticks(np.arange(-0.5, len(P_VALUES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(T_CUT_VALUES), 1), minor=True)
    ax.grid(which="minor", color=SURFACE, linewidth=2)
    ax.tick_params(which="minor", length=0)

    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]
            resolved = True if mask is None else bool(mask[row, col])
            label = fmt.format(value) if resolved else f"({fmt.format(value)})"
            color = (_text_color_for(im.cmap(im.norm(value))) if resolved else INK_MUTED)
            ax.text(col, row, label, ha="center", va="center",
                     color=color, fontsize=8, alpha=1.0 if resolved else 0.75)

    return im


def draw_figure(matrix, label, fmt="{:.2f}", mask=None, vmax=None):
    """`vmax` fixes the colour scale; pass the same value to every panel of a
    sweep so their colours are comparable."""
    if vmax is None:
        vmax = float(np.nanmax(np.abs(matrix)))
    vmax = vmax or 1e-9
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(7.5, 6.5), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    im = draw_heatmap(ax, matrix, norm, fmt=fmt, mask=mask)

    cbar = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.03)
    cbar.set_label(label, color=INK_SECONDARY)
    cbar.ax.yaxis.set_tick_params(color=INK_MUTED, labelcolor=INK_MUTED)
    cbar.outline.set_visible(False)

    plt.show()


# One scale for the whole sweep, so panel-to-panel colour differences are real.
VMAX = float(np.nanmax(np.abs(ratio)))

for k, p_s in enumerate(P_S_VALUES):
    print(f"p_s = {p_s:.1f}")
    draw_figure(
        ratio[k], "(T_swap-asap − T_cutoff-aware) / T_cutoff-aware",
        mask=significant[k], vmax=VMAX,
    )

## Notes

_Populated after the run below._